In [4]:
import urllib.request, hashlib, zipfile

url = "https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip"
urllib.request.urlretrieve(url, "data.zip")

with open("data.zip", "rb") as f:
    print(hashlib.sha256(f.read()).hexdigest())

with zipfile.ZipFile("data.zip") as z:
    z.extractall(".")

9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67


In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")
print(torch.__version__)

2.15.0+cpu


In [3]:
evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder("data/train", transform=evaluation_transform)
evaluation_dataset = datasets.ImageFolder("data/test", transform=evaluation_transform)

In [4]:
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [5]:
print(len(train_dataset), len(evaluation_dataset))
print(train_dataset.class_to_idx)

800 201
{'curly': 0, 'straight': 1}


In [6]:
import torch.nn as nn


class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

In [8]:
model = HairModel().to(device)
print(sum(p.numel() for p in model.parameters()))

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

20073473


In [9]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            labels = labels.float().unsqueeze(1)
            logits = model(images)
            loss = criterion(logits, labels)
            loss_sum += loss.item() * images.size(0)
            correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()
            total += labels.size(0)
    return loss_sum / total, correct / total

def train_phase(model, loader, epochs=10):
    history = {"train_loss": [], "train_accuracy": [],
               "evaluation_loss": [], "evaluation_accuracy": []}
    for epoch in range(epochs):
        model.train()
        loss_sum, correct, total = 0.0, 0, 0
        for images, labels in loader:
            labels = labels.float().unsqueeze(1)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * images.size(0)
            correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()
            total += labels.size(0)
        ev_loss, ev_acc = evaluate(model, evaluation_loader)
        history["train_loss"].append(loss_sum / total)
        history["train_accuracy"].append(correct / total)
        history["evaluation_loss"].append(ev_loss)
        history["evaluation_accuracy"].append(ev_acc)
        print(f"epoch {epoch+1}: train_loss={loss_sum/total:.3f} "
              f"train_acc={correct/total:.3f} eval_acc={ev_acc:.3f}")
    return history

In [10]:
import json
history_baseline = train_phase(model, train_loader, epochs=10)
json.dump(history_baseline, open("history_baseline.json", "w"))

epoch 1: train_loss=0.639 train_acc=0.645 eval_acc=0.657
epoch 2: train_loss=0.544 train_acc=0.718 eval_acc=0.652
epoch 3: train_loss=0.496 train_acc=0.755 eval_acc=0.622
epoch 4: train_loss=0.433 train_acc=0.791 eval_acc=0.692
epoch 5: train_loss=0.405 train_acc=0.809 eval_acc=0.662
epoch 6: train_loss=0.395 train_acc=0.807 eval_acc=0.711
epoch 7: train_loss=0.344 train_acc=0.836 eval_acc=0.672
epoch 8: train_loss=0.281 train_acc=0.885 eval_acc=0.721
epoch 9: train_loss=0.224 train_acc=0.910 eval_acc=0.716
epoch 10: train_loss=0.158 train_acc=0.939 eval_acc=0.637


In [11]:
print(round(float(np.median(history_baseline["train_accuracy"])), 2))
print(round(float(np.std(history_baseline["train_loss"], ddof=0)), 3))

0.81
0.139


In [12]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(50, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1),
                                 interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset_aug = datasets.ImageFolder("data/train", transform=train_transform_augmented)
train_loader_aug = DataLoader(train_dataset_aug, batch_size=20, shuffle=True, num_workers=0,
                              generator=torch.Generator().manual_seed(SEED + 1))

history_augmented = train_phase(model, train_loader_aug, epochs=10)
json.dump(history_augmented, open("history_augmented.json", "w"))

epoch 1: train_loss=0.676 train_acc=0.605 eval_acc=0.672
epoch 2: train_loss=0.613 train_acc=0.647 eval_acc=0.627
epoch 3: train_loss=0.590 train_acc=0.680 eval_acc=0.711
epoch 4: train_loss=0.592 train_acc=0.675 eval_acc=0.711
epoch 5: train_loss=0.560 train_acc=0.714 eval_acc=0.721
epoch 6: train_loss=0.557 train_acc=0.708 eval_acc=0.701
epoch 7: train_loss=0.543 train_acc=0.700 eval_acc=0.741
epoch 8: train_loss=0.532 train_acc=0.729 eval_acc=0.706
epoch 9: train_loss=0.504 train_acc=0.756 eval_acc=0.721
epoch 10: train_loss=0.540 train_acc=0.705 eval_acc=0.731


In [14]:
print(round(float(np.mean(history_augmented["evaluation_loss"])), 3))
print(round(float(np.mean(history_augmented["evaluation_accuracy"][-5:])), 2))

0.576
0.72
